<br/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="left"/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="right"/>
<div align="center">
<h2>Bootcamp Data Science — Módulo 2</h2><br/>
<h1>Semana 5 · Viernes — KNN Regresor y Árbol de Decisión</h1>
<h3>Modelos no-lineales para regresión + Primer benchmark</h3>
<br/>
    <b>Instructor:</b> Jesús Ortiz · jesus.jeduardo7@gmail.com<br/><br/>
    <b>SkillNest · b2b-sonda-data-science</b>
</div>
<br/>

## 🎯 Objetivos de hoy

Al final vas a poder:

1. Entender la intuición de **KNN para regresión** y el rol del parámetro `n_neighbors`.
2. Entender la intuición de **árboles de decisión** y los parámetros `max_depth`, `min_samples_leaf`.
3. Saber **cuándo usar cada modelo** y por qué ninguno gana siempre.
4. Hacer tu **primer benchmark** comparando 3 modelos sobre el mismo dataset.
5. Resolver **3 ejercicios desafiantes** con datasets nuevos.

> Requisito previo: clases del lunes (estandarización + train/test), martes (regresión lineal + métricas), miércoles (encoding).

# 1. KNN Regresor — "dime quién es tu vecino"

## La idea en una frase

**KNN regresor** predice el valor de un punto nuevo **promediando el valor de los `k` vecinos más cercanos** del set de entrenamiento.

### 🧠 Analogía simple

Imagina que quieres estimar el precio de una casa nueva. Vas a Zillow, buscas las **5 casas más parecidas a la tuya** (mismo barrio, mismo tamaño, mismas habitaciones), miras sus precios y **calculas el promedio**. Eso es exactamente lo que hace KNN con `k=5`.

## ¿Qué significa "más cercanos"?

KNN mide **distancia** entre puntos en el espacio de features. Por defecto usa **distancia euclidiana**. Por eso:

> ⚠️ **KNN exige estandarizar SIEMPRE.** Si una columna está en miles y otra en unidades, la primera dominará la distancia. Sin estandarización KNN funciona terrible.

## Parámetro principal: `n_neighbors`

| `n_neighbors` | Comportamiento |
|---|---|
| `k=1` | Predice exactamente el valor del vecino más cercano → **muy sensible al ruido** |
| `k` pequeño (3-5) | Modelo más "local", capta variaciones pero puede sobreajustar |
| `k` grande (20+) | Modelo más "suave", promedia mucho — puede subajustar |
| `k = n_train` | Predice siempre el promedio total → modelo inútil |

**Regla práctica:** empezar con `k=5` y probar 3, 7, 11, 15 para ajustar.

## 👀 Demo en vivo — KNN con dataset `mpg`

Vamos a predecir el consumo de combustible (`mpg`) de autos a partir de sus características.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Dataset: autos con su consumo de combustible
mpg = sns.load_dataset('mpg').dropna()
print(f'Shape: {mpg.shape}')
mpg.head()

In [ ]:
# Solo features numéricas (sin origin y name por simplicidad)
X = mpg.drop(columns=['mpg', 'origin', 'name'])
y = mpg['mpg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# OBLIGATORIO: estandarizar para KNN
sc = StandardScaler()
X_train_esc = sc.fit_transform(X_train)
X_test_esc  = sc.transform(X_test)

print(f'Features: {list(X.columns)}')
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Entrenar con k=5 por defecto
modelo_knn = KNeighborsRegressor(n_neighbors=5)
modelo_knn.fit(X_train_esc, y_train)

y_pred = modelo_knn.predict(X_test_esc)

print(f'R²:   {r2_score(y_test, y_pred):.4f}')
print(f'MAE:  {mean_absolute_error(y_test, y_pred):.2f} mpg')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f} mpg')

In [ ]:
# ¿Cómo cambia el R² con distintos k?
ks = [1, 3, 5, 7, 11, 15, 21, 31, 51]
scores = []

for k in ks:
    m = KNeighborsRegressor(n_neighbors=k).fit(X_train_esc, y_train)
    scores.append(m.score(X_test_esc, y_test))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ks, scores, marker='o', linewidth=2, color='steelblue')
ax.set_xlabel('n_neighbors (k)')
ax.set_ylabel('R² en test')
ax.set_title('Efecto de k en el rendimiento de KNN')
ax.grid(alpha=0.3)
plt.show()

mejor_k = ks[np.argmax(scores)]
print(f'👉 Mejor k: {mejor_k} con R² = {max(scores):.4f}')
print('   Notar cómo k=1 suele ser peor (ruidoso) y k muy grande también baja (subajuste).')

# 2. Árbol de Decisión para Regresión

## La idea en una frase

Un **árbol de decisión** hace **preguntas binarias** sobre las features hasta llegar a un valor predicho.

### 🧠 Analogía simple

Imagina que estás adivinando el precio de una casa con un amigo. Él te va guiando con preguntas:

- *"¿La casa tiene más de 3 habitaciones?"* → Si NO → predice $150k. Si SÍ →
- *"¿Está en zona costera?"* → Si NO → predice $250k. Si SÍ →
- *"¿Tiene más de 200m²?"* → Si NO → predice $350k. Si SÍ → predice $500k.

Eso es exactamente un árbol: una secuencia de preguntas binarias que terminan en un valor.

## Parámetros importantes

| Parámetro | Para qué sirve |
|---|---|
| `max_depth` | Profundidad máxima del árbol. **Sin límite = overfitting**. Típico: 3-10 |
| `min_samples_split` | Mínimo de muestras para hacer un nuevo split. Default 2 |
| `min_samples_leaf` | Mínimo de muestras en una hoja final |
| `random_state` | Para reproducibilidad |

## ¿Por qué los árboles NO necesitan estandarización?

Porque solo hacen comparaciones del tipo *"feature > umbral"* — la escala no afecta. Esto los hace muy cómodos.

## 👀 Demo en vivo — Árbol con `mpg`

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

# Mismo split, pero ahora SIN estandarizar (árboles no lo necesitan)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo_arbol = DecisionTreeRegressor(max_depth=4, random_state=42)
modelo_arbol.fit(X_train, y_train)

y_pred = modelo_arbol.predict(X_test)

print(f'R²:   {r2_score(y_test, y_pred):.4f}')
print(f'MAE:  {mean_absolute_error(y_test, y_pred):.2f} mpg')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f} mpg')

In [ ]:
# 🌳 Visualizar el árbol — esta es una de las grandes ventajas
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(modelo_arbol, 
          feature_names=X.columns, 
          filled=True, rounded=True, fontsize=9,
          ax=ax)
plt.title('Árbol de decisión para predecir MPG (profundidad máx = 4)', fontsize=14)
plt.show()

In [ ]:
# ⚠️ El overfitting de los árboles — ver cómo cambia el R² según max_depth
depths = [1, 2, 3, 5, 7, 10, 15, 20, None]
scores_train, scores_test = [], []

for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train)
    scores_train.append(m.score(X_train, y_train))
    scores_test.append(m.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(10, 5))
x_labels = [str(d) if d is not None else 'sin límite' for d in depths]
ax.plot(x_labels, scores_train, marker='o', linewidth=2, color='#70AD47', label='R² Train')
ax.plot(x_labels, scores_test, marker='o', linewidth=2, color='#C0504D', label='R² Test')
ax.set_xlabel('max_depth')
ax.set_ylabel('R²')
ax.set_title('Overfitting: el train sube siempre, el test eventualmente baja')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print('👉 Mira cómo el R² de train llega a casi 1.0 con árboles muy profundos.')
print('   Pero el R² de test EMPIEZA A BAJAR — eso es overfitting.')
print('   El truco está en encontrar el max_depth óptimo (típicamente 4-7).')

# 3. Comparación rápida: Lineal vs KNN vs Árbol

Tres modelos compitiendo sobre el mismo dataset `mpg`.

In [ ]:
from sklearn.linear_model import LinearRegression

# Lineal — necesita estandarización
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
X_train_esc = sc.fit_transform(X_train)
X_test_esc  = sc.transform(X_test)

modelos = {
    'Regresión Lineal':      (LinearRegression(),               X_train_esc, X_test_esc),
    'KNN Regresor (k=7)':    (KNeighborsRegressor(n_neighbors=7), X_train_esc, X_test_esc),
    'Árbol (max_depth=5)':   (DecisionTreeRegressor(max_depth=5, random_state=42), X_train, X_test),
}

resultados = []
for nombre, (modelo, Xtr, Xte) in modelos.items():
    modelo.fit(Xtr, y_train)
    pred = modelo.predict(Xte)
    resultados.append({
        'Modelo': nombre,
        'R²': r2_score(y_test, pred),
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred))
    })

tabla = pd.DataFrame(resultados).set_index('Modelo').round(3)
tabla

### Conclusión visual

En `mpg`, los 3 modelos suelen rendir parecido (R² ≈ 0.80-0.85). Pero en datasets con relaciones **no-lineales fuertes**, KNN y árboles superan a la regresión lineal.

> 💡 **No hay modelo ganador universal.** Por eso siempre comparamos varios — eso es **benchmarking**.

---
# 🏋️ Ejercicios prácticos — 3 desafíos

## Ejercicio 1 — KNN con el dataset `diamonds` 💎

Vamos a predecir el **precio de un diamante** usando KNN.

**Dataset:** `sns.load_dataset('diamonds')` — 53,940 diamantes con sus características.

**Tarea:**

1. Cargar el dataset.
2. Codificar las **3 ordinales** con `OrdinalEncoder` respetando el orden:
   - `cut`: Fair < Good < Very Good < Premium < Ideal
   - `color`: J < I < H < G < F < E < D (D es el mejor)
   - `clarity`: I1 < SI2 < SI1 < VS2 < VS1 < VVS2 < VVS1 < IF
3. Definir `y = price` y `X = el resto`.
4. Train/test split 80/20, `random_state=42`.
5. **OBLIGATORIO:** estandarizar (KNN usa distancias).
6. Probar **3 valores de `k`**: 5, 15, 30. ¿Cuál gana?
7. Reportar R², MAE y RMSE del mejor k.

**Pista:** este dataset es grande (53k filas) — KNN puede tardar 10-20 segundos en predecir.

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = sns.load_dataset('diamonds').copy()

# 1. Codificación ordinal de las 3 categóricas
encoder = OrdinalEncoder(categories=[
    ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal'],
    ['J', 'I', 'H', 'G', 'F', 'E', 'D'],
    ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']
])
df[['cut', 'color', 'clarity']] = encoder.fit_transform(df[['cut', 'color', 'clarity']])

# 2. X / y
X = df.drop(columns=['price'])
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Estandarizar (CLAVE para KNN)
sc = StandardScaler()
X_train_esc = sc.fit_transform(X_train)
X_test_esc  = sc.transform(X_test)

# 4. Probar 3 valores de k
for k in [5, 15, 30]:
    m = KNeighborsRegressor(n_neighbors=k).fit(X_train_esc, y_train)
    r2 = m.score(X_test_esc, y_test)
    print(f'k={k:3d} → R²={r2:.4f}')

# 5. Métricas con el mejor (suele ser k=5 o k=15)
best = KNeighborsRegressor(n_neighbors=5).fit(X_train_esc, y_train)
y_pred = best.predict(X_test_esc)
print(f'\nMejor modelo (k=5):')
print(f'  R²:   {r2_score(y_test, y_pred):.4f}')
print(f'  MAE:  ${mean_absolute_error(y_test, y_pred):.0f}')
print(f'  RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred)):.0f}')
# 👉 KNN obtiene R² ≈ 0.97 — los diamantes muy parecidos cuestan muy parecido.
```
</details>

## Ejercicio 2 — Árbol con `load_diabetes` 🩺

Vamos a predecir la **progresión de la diabetes** un año después del diagnóstico.

**Dataset:** `from sklearn.datasets import load_diabetes` — 442 pacientes, 10 features médicas ya estandarizadas.

**Tarea:**

1. Cargar el dataset:
   ```python
   from sklearn.datasets import load_diabetes
   data = load_diabetes(as_frame=True)
   X, y = data.data, data.target
   ```
2. Train/test split 80/20, `random_state=42`. Los datos ya están escalados — **no necesitas estandarizar**.
3. Entrenar un **árbol con `max_depth=3`** y reportar R² de **train Y test**.
4. Repetir con `max_depth=5, 10, sin límite`. ¿En cuál hay más overfitting?
5. Visualizar el árbol con `max_depth=3` (con `plot_tree`).
6. **Interpretar:** ¿cuál es la primera variable que el árbol usa para dividir? ¿Por qué tiene sentido (puedes ver el significado de las features en `data.DESCR`)?

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree

data = load_diabetes(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Probar varios max_depth
for d in [3, 5, 10, None]:
    m = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train)
    print(f'max_depth={str(d):8s} → R² train={m.score(X_train, y_train):.3f}  R² test={m.score(X_test, y_test):.3f}')

# Árbol max_depth=3 visualizado
modelo = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_train, y_train)
fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(modelo, feature_names=X.columns, filled=True, rounded=True, fontsize=10, ax=ax)
plt.show()

# 👉 La primera variable que aparece suele ser 'bmi' (índice de masa corporal) o 's5'
#    (nivel de triglicéridos). Ambas tienen sentido médico — son indicadores
#    fuertes de progresión de diabetes.
#
# 👉 Con max_depth=None, R² train ≈ 1.0 pero R² test ~0.15 → overfitting masivo.
```
</details>

## Ejercicio 3 — BENCHMARK: 3 modelos vs California Housing 🏠

Vamos a hacer nuestro primer **benchmark serio**: comparar Regresión Lineal, KNN y Árbol sobre un dataset nuevo y desafiante.

**Dataset:** `fetch_california_housing` de sklearn — 20,640 distritos de California con 8 features (NO es el csv que ya usaron — viene directo de sklearn).

**Tarea:**

1. Cargar el dataset:
   ```python
   from sklearn.datasets import fetch_california_housing
   data = fetch_california_housing(as_frame=True)
   X, y = data.data, data.target  # y = valor mediano de casa (en cientos de miles USD)
   ```
2. Train/test 80/20, `random_state=42`.
3. Estandarizar (lo necesitan Lineal y KNN; al Árbol no le afecta).
4. Entrenar **3 modelos**:
   - `LinearRegression()`
   - `KNeighborsRegressor(n_neighbors=10)`
   - `DecisionTreeRegressor(max_depth=8, random_state=42)`
5. Construir un **DataFrame de comparación** con columnas: `Modelo, R², MAE, RMSE`.
6. Hacer un **barplot** comparando los R² de los 3 modelos.
7. **Conclusión escrita:** ¿qué modelo ganó? ¿Tiene sentido? (los precios de casas tienen relaciones no-lineales por geografía e ingreso).

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Datos
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target

# 2. Split + estandarización
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
X_train_esc = sc.fit_transform(X_train)
X_test_esc  = sc.transform(X_test)

# 3. Entrenar y evaluar
modelos = {
    'Regresión Lineal':    (LinearRegression(),                               X_train_esc, X_test_esc),
    'KNN (k=10)':          (KNeighborsRegressor(n_neighbors=10),              X_train_esc, X_test_esc),
    'Árbol (depth=8)':     (DecisionTreeRegressor(max_depth=8, random_state=42), X_train, X_test),
}

resultados = []
for nombre, (modelo, Xtr, Xte) in modelos.items():
    modelo.fit(Xtr, y_train)
    pred = modelo.predict(Xte)
    resultados.append({
        'Modelo': nombre,
        'R²':   r2_score(y_test, pred),
        'MAE':  mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred))
    })

df_resultados = pd.DataFrame(resultados).set_index('Modelo').round(4)
print(df_resultados)

# 4. Barplot de R²
fig, ax = plt.subplots(figsize=(9, 5))
colores = ['#4472C4', '#ED7D31', '#70AD47']
ax.bar(df_resultados.index, df_resultados['R²'], color=colores, edgecolor='white')
for i, v in enumerate(df_resultados['R²']):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
ax.set_title('Benchmark — California Housing (R² en test)', fontsize=13, fontweight='bold')
ax.set_ylabel('R²')
ax.set_ylim(0, max(df_resultados['R²']) * 1.15)
ax.grid(axis='y', alpha=0.3)
plt.show()

# 👉 Conclusión típica:
#    - KNN y Árbol superan a la regresión lineal (R² ~0.70 vs ~0.61).
#    - Tiene sentido: la geografía (latitud/longitud) y los ingresos tienen 
#      relaciones NO-LINEALES con el precio.
#    - La regresión lineal asume linealidad — por eso pierde aquí.
```
</details>

## Ejercicio 4 — Tuning de hiperparámetros con taxis 🚖

Vamos a predecir el **precio total de un viaje en taxi en NYC** y, sobre todo, vamos a aprender a **buscar los mejores hiperparámetros** de cada modelo.

**Dataset:** `sns.load_dataset('taxis')` — 6,433 viajes reales de NYC con distancia, tarifa, tipo de pago, boroughs de origen/destino y color del taxi (yellow / green).

**Tarea:**

1. **Cargar y limpiar:**
   ```python
   taxis = sns.load_dataset('taxis').dropna()
   ```
2. **Definir target y features:**
   - Target: `total` (precio total del viaje en USD)
   - Features útiles: `distance`, `passengers`, `pickup_borough`, `dropoff_borough`, `payment`, `color`
   - ⚠️ **NO incluyas `fare`, `tip` ni `tolls`** — son componentes directos del total (sería trampa)
3. **Codificar las categóricas** con `pd.get_dummies()` (`drop_first=True`).
4. Train/test split 80/20, `random_state=42`. Estandarizar (lo necesita KNN).
5. **Tuning de KNN — probar varios `k`:**
   - Loop sobre `k = [3, 5, 7, 10, 15, 20, 30]`
   - Guardar R² de test para cada uno
   - Graficar R² vs k y elegir el **mejor k**
6. **Tuning de Árbol — probar varios `max_depth`:**
   - Loop sobre `max_depth = [3, 5, 7, 10, 15, None]`
   - Guardar R² de **train Y test** para cada uno
   - Graficar ambas curvas → identificar dónde empieza el overfitting
7. **Decisión final:** ¿qué modelo eliges y con qué hiperparámetro? Reporta R², MAE y RMSE finales.

**Pista:** este es el **flujo de trabajo real** de un data scientist — no se entrena un solo modelo, se prueban varios y se elige el mejor con base en métricas en test.

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Cargar y limpiar
taxis = sns.load_dataset('taxis').dropna()

# 2. Features y target (SIN fare, tip ni tolls)
features = ['distance', 'passengers', 'pickup_borough', 'dropoff_borough', 'payment', 'color']
df = taxis[features + ['total']].copy()

# 3. Codificar categóricas
df_cod = pd.get_dummies(df, drop_first=True, dtype=int)

X = df_cod.drop(columns=['total'])
y = df_cod['total']

# 4. Split + estandarizar
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
X_train_esc = sc.fit_transform(X_train)
X_test_esc  = sc.transform(X_test)

# 5. TUNING KNN
ks = [3, 5, 7, 10, 15, 20, 30]
scores_knn = [KNeighborsRegressor(n_neighbors=k).fit(X_train_esc, y_train).score(X_test_esc, y_test) for k in ks]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(ks, scores_knn, marker='o', linewidth=2, color='#4472C4')
axes[0].set_xlabel('n_neighbors (k)'); axes[0].set_ylabel('R² en test')
axes[0].set_title('Tuning de KNN'); axes[0].grid(alpha=0.3)

mejor_k = ks[np.argmax(scores_knn)]
print(f'KNN → mejor k = {mejor_k}, R² = {max(scores_knn):.4f}')

# 6. TUNING ÁRBOL — train Y test
depths = [3, 5, 7, 10, 15, None]
tr, te = [], []
for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train)
    tr.append(m.score(X_train, y_train))
    te.append(m.score(X_test, y_test))

x_labels = [str(d) if d else 'None' for d in depths]
axes[1].plot(x_labels, tr, marker='o', linewidth=2, color='#70AD47', label='Train')
axes[1].plot(x_labels, te, marker='o', linewidth=2, color='#C0504D', label='Test')
axes[1].set_xlabel('max_depth'); axes[1].set_ylabel('R²')
axes[1].set_title('Tuning de Árbol — ¿dónde empieza el overfitting?')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

mejor_depth = depths[np.argmax(te)]
print(f'Árbol → mejor max_depth = {mejor_depth}, R² test = {max(te):.4f}')

# 7. DECISIÓN FINAL — entrenar el ganador
if max(scores_knn) > max(te):
    print(f'\n🏆 GANADOR: KNN con k={mejor_k}')
    final = KNeighborsRegressor(n_neighbors=mejor_k).fit(X_train_esc, y_train)
    pred = final.predict(X_test_esc)
else:
    print(f'\n🏆 GANADOR: Árbol con max_depth={mejor_depth}')
    final = DecisionTreeRegressor(max_depth=mejor_depth, random_state=42).fit(X_train, y_train)
    pred = final.predict(X_test)

print(f'\nMétricas finales:')
print(f'  R²:   {r2_score(y_test, pred):.4f}')
print(f'  MAE:  ${mean_absolute_error(y_test, pred):.2f}')
print(f'  RMSE: ${np.sqrt(mean_squared_error(y_test, pred)):.2f}')

# 👉 Insight clave: el árbol y KNN se comportan muy distinto con la complejidad.
#    El árbol con max_depth=None llega a R² train ≈ 1.0 pero overfittea.
#    KNN con k muy pequeño también overfittea, con k muy grande subajusta.
#    EL ARTE ESTÁ EN ENCONTRAR EL PUNTO MEDIO.
```
</details>

---
## 📌 Cierre del día

Hoy aprendimos:

- ✅ **KNN Regresor** — promedia los `k` vecinos más cercanos. **Estandarizar es obligatorio.**
- ✅ **Árbol de Decisión** — preguntas binarias hasta llegar a un valor. No necesita estandarización.
- ✅ El parámetro `n_neighbors` en KNN y `max_depth` en árboles controlan el **overfitting**.
- ✅ Los árboles tienen una gran ventaja: **se pueden visualizar y son explicables**.
- ✅ **Benchmark**: comparar varios modelos sobre el mismo dataset → no hay ganador universal.

### 🔜 Próxima semana — Semana 6

- **Lunes 25:** Pipelines y ColumnTransformer
- **Martes 26:** SimpleImputer y guardar modelos
- **Miércoles 27:** Random Forest y Bagging para regresión
- **Jueves 28:** Benchmarking avanzado
- **Viernes 29:** Proyecto integrador de regresión

Nos vemos 🚀